[<img src='https://github.com/jeshraghian/snntorch/blob/master/docs/_static/img/snntorch_alpha_w.png?raw=true' width="300">](https://github.com/jeshraghian/snntorch)

# Introduction to delta neurons in spiking neural networks
### Tutorial written by Danny Chia and Jason You

<!-- TO DO: link to repository once code is merged into the stable release -->
<!--
<a href="https://colab.research.google.com/github/jeshraghian/snntorch/blob/master/examples/quickstart.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>
-->

[<img src='https://github.com/jeshraghian/snntorch/blob/master/docs/_static/img/GitHub-Mark-Light-120px-plus.png?raw=true' width="28">](https://github.com/jeshraghian/snntorch) [<img src='https://github.com/jeshraghian/snntorch/blob/master/docs/_static/img/GitHub_Logo_White.png?raw=true' width="80">](https://github.com/jeshraghian/snntorch)

## Introduction

Unlike traditional artificial neural networks, spiking neural networks (SNNs) are driven by events and only process data when a spike is detected. Standard SNN frameworks such as snnTorch typically adopt the LIF model at the core: spikes are generated when the membrane potential exceeds a specific threshold. However, many events occurring in nature are more accurately described as changes over time rather than magnitudes.

## Background

SNN frameworks typically adopt LIF models, which are a computationally efficient abstraction of neural dynamics. LIF models generate spikes based on the absolute potential of the membrane; this means the neuron will spike when the input currents cross the static threshold $ U_{thr} $:

$$ \begin{equation}
  S_t = \begin{cases}
    1, & U_t > U_{thr} \\
    0, & \text{otherwise}.
  \end{cases}
\end{equation} $$

Once the neuron spikes, the membrane potential will be reset to zero. The LIF neuron is represented by the equation:

$$ U_{t + 1} = {βU_t} + {WX_{t + 1}} - R_t $$

Here, *β* is the decay factor. It is multiplied by the current potential $ {U_t} $ to yield the leaky component $ {βU_t} $ of the neuron. The synaptic weight $ W $ is a learnable parameter, and $ {X_{t + 1}} $ is an input spike at time *t* + 1; they are multiplied together to produce the input current $ {WX_{t + 1}} $ at the next time step. $ {R_t} $ represents the reset mechanism that either reduces the potential by the threshold value or sets it to zero, depending on the configuration. While effective, this can be very energy inefficient due to repeated spiking during sustained inputs.<sup>[2]</sup>

As an alternative to traditional SNNs, a delta network computes the difference between the current state and the last transmitted state, and updates are sent only if the difference exceeds the predefined threshold. Because this approach ignores minor fluctuations, noise and redundant spiking activity are suppressed. This allows reduced communication costs while keeping task-relevant information, demonstrating both computational and energy efficiency advantages. Delta networks compute state changes as follows:

$$ Δ {x_t} = {x_t} - {x_{t - 1}} $$

$$ \begin{equation}
  E_t = \begin{cases}
    Δ x_t, & |\Delta x_t| \geq Θ_Δ \\
    0, & \text{otherwise}.
  \end{cases}
\end{equation} $$

$ {x_t} $ is the current network state, $ {x_{t - 1}} $ is the last transmitted state, and $ {Θ_Δ} $ is the threshold that controls the sensitivity to state changes.<sup>[1]</sup> Delta networks also support negative spikes, which can reduce the membrane potential and prevent excessive firing of spikes from previous time steps. This can help reduce latency as well as achieve greater information capacity compared to standard LIF neurons.

## About the `DeltaLeaky` class

We keep the standard equation for updating the membrane potential but introduce a variable $ {U_Δ} $ for storing the change in potential:

$$ {U_Δ = U_{t + 1} - U_t} $$

In this case, the neuron fires a spike when the change exceeds a predefined threshold:

$$ \begin{equation}
  S_t = \begin{cases}
    1, & U_Δ \geq \Delta_{thr} \\
    0, & \text{otherwise}.
  \end{cases}
\end{equation} $$

The above is implemented in the modified `forward()` function from the `leaky.py` class. This tutorial demonstrates how the `DeltaLeaky` neuron type can be used.

## 0. Set up environment

Download and install snnTorch package.

In [ ]:
!git clone -b feature/delta-SNNs https://github.com/ixfd64/snntorch.git
%cd snntorch
!pip install -e .

# TO DO: change to pip once code is merged into the stable branch
# !pip install snntorch --quiet

Import required libraries.

In [ ]:
import torch
import torch.nn as nn
import snntorch as snn
import matplotlib.pyplot as plt

from snntorch import utils

## 1. Define parameters, input current and helper function

Define parameters and the input current to be passed to the neuron. Instead of a standard `threshold` parameter, delta neurons use a `delta_threshold` parameter for the *change* in potential.

In [ ]:
num_steps = 100     # time steps
beta = 0.8          # membrane potential decay rate
amplitude = 1.2     # input current amplitude

# simulate input current
cur_in = torch.ones(num_steps) * 0.5
# ensure current exceeds the threshold after some integration time
cur_in[num_steps // 2:] = amplitude

Define a function to process the input and membrane potential.

In [ ]:
def simulate_neuron(neuron):

  # initialize arrays to store membrane potential and spikes
  mem_rec = []
  spk_rec = []

  # initial reset
  utils.reset(neuron)

  # simulate the neuron over time
  for step in range(num_steps):
      # process current input and previous membrane potential
      spk_out, mem_out = neuron(cur_in[step])

      # store outputs
      if isinstance(neuron, snn.DeltaLeaky):
        mem_rec.append(mem_out[0])
      else:
        mem_rec.append(mem_out)

      spk_rec.append(spk_out)

  # convert output lists to tensors
  mem_rec = torch.stack(mem_rec)
  spk_rec = torch.stack(spk_rec)

  return mem_rec, spk_rec

## 2. Initialize neurons

We initialize both a standard spiking neuron and a delta neuron for comparision purposes.

In [ ]:
# initialize standard neuron
leaky_neuron = snn.Leaky(beta=beta, threshold=0.3, init_hidden=True, output=True)

# initialize delta neuron
delta_neuron = snn.DeltaLeaky(beta=beta, delta_threshold=0.3, init_hidden=True, output=True)

leaky_mem, leaky_spikes = simulate_neuron(leaky_neuron)
delta_mem, delta_spikes = simulate_neuron(delta_neuron)

## 3. Visualize the data

Plot the input current over time.

In [ ]:
_fig1, ax1 = plt.subplots()
ax1.plot(cur_in)
ax1.set_title("Input current")
ax1.set_xlabel('t')
ax1.set_ylabel("I")
plt.show()

We generate a standard leaky neuron for reference.

In [ ]:
_fig2, ax2 = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# plot membrane potential
ax2[0].plot(leaky_mem.detach().numpy())
ax2[0].set_title('Potential')
ax2[0].set_xlabel('t')
ax2[0].set_ylabel('U')
ax2[0].axhline(y=leaky_neuron.threshold, color='r', linestyle='--', label='Threshold')
ax2[0].legend()

# plot spikes
ax2[1].scatter(range(num_steps), leaky_spikes.detach().numpy(), color='orange')
ax2[1].set_title('Output spikes')
ax2[1].set_xlabel('t')
ax2[1].set_ylabel('Spike')
ax2[1].set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()


Compare this with the new `DeltaLeaky` neuron.

In [ ]:
_fig3, ax3 = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# plot membrane potential
ax3[0].plot(delta_mem.detach().numpy())
ax3[0].set_title('Potential')
ax3[0].set_xlabel('t')
ax3[0].set_ylabel('U')

# plot spikes
ax3[1].scatter(range(num_steps), delta_spikes.detach().numpy(), color='orange')
ax3[1].set_title('Output spikes')
ax3[1].set_xlabel('t')
ax3[1].set_ylabel('Spike')
ax3[1].set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()


Spikes should not be generated when the input current is below the delta threshold. Let's set `delta_threshold` to 1 and see what happens.

In [ ]:
delta_neuron_hi = snn.DeltaLeaky(beta=beta, delta_threshold=1.0, init_hidden=True, output=True)

delta_mem_hi, delta_spikes_hi = simulate_neuron(delta_neuron_hi)

_fig4, ax4 = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# plot membrane potential
ax4[0].plot(delta_mem.detach().numpy())
ax4[0].set_title('Potential')
ax4[0].set_xlabel('t')
ax4[0].set_ylabel('U')

# plot spikes
ax4[1].scatter(range(num_steps), delta_spikes_hi.detach().numpy(), color='orange')
ax4[1].set_title('Output spikes')
ax4[1].set_xlabel('t')
ax4[1].set_ylabel('Spike')
ax4[1].set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()


## 4. Train on a sample data set

We train on the MNIST data set to demonstrate that `DeltaLeaky` neurons can be used for machine learning tasks.

In [ ]:
# define data loading parameters
batch_size = 128
data_path = '/tmp/data/mnist'
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

Define a transform and load the training and test sets.

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# define a transform
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(),
    transforms.ToTensor(),
    transforms.Normalize((0,), (1,))
    ])

mnist_train = datasets.MNIST(data_path, train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(data_path, train=False, download=True, transform=transform)

# create data loaders
train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=True)

Set up a neural network with two linear and two `DeltaLeaky` layers.

In [ ]:
import torch.nn.functional as F

class Net(nn.Module):
  def __init__(self):
    super().__init__()

    num_inputs = 784  # number of inputs
    num_hidden = 300  # number of hidden neurons
    num_outputs = 10  # number of classes (output neurons)

    # static global decay rate for all leaky neurons in layer 1
    beta1 = 0.9
    # learnable independent decay rate for each leaky neuron in layer 2: [0, 1)
    beta2 = torch.rand((num_outputs), dtype = torch.float)

    # initialize layers
    self.fc1 = nn.Linear(num_inputs, num_hidden)
    self.delta_lif1 = snn.DeltaLeaky(beta=beta1)
    self.fc2 = nn.Linear(num_hidden, num_outputs)
    self.delta_lif2 = snn.DeltaLeaky(beta=beta2, learn_beta=True)

  def forward(self, x):
    # reset hidden states
    mem1 = self.delta_lif1.reset_mem()
    mem2 = self.delta_lif2.reset_mem()
    # record output spikes and hidden states
    spk2_rec = []
    mem2_rec = []

    # loop over time
    for step in range(num_steps):
      cur1 = self.fc1(x.flatten(1))
      spk1, mem1 = self.delta_lif1(cur1, mem1)
      cur2 = self.fc2(spk1)
      spk2, mem2 = self.delta_lif2(cur2, mem2)

      # record spikes and membrane potential
      spk2_rec.append(spk2)
      mem2_rec.append(mem2[0])

    return torch.stack(spk2_rec), torch.stack(mem2_rec)

# use CUDA if available
net = Net().to(device)

Define a forward pass function to feed data through the SNN.

In [ ]:
def forward_pass(net, data, num_steps):
  spk_rec = []  # record spikes over time

  # loop over time
  for step in range(num_steps):
    # one time step of the forward-pass
    spk_out, mem_out = net(data)
    # record spikes
    spk_rec.append(spk_out)

  return torch.stack(spk_rec)

Train a model on the MNIST data sets.

In [ ]:
import snntorch.functional as SF

optimizer = torch.optim.Adam(net.parameters(), lr=2e-3, betas=(0.9, 0.999))
loss_fn = SF.mse_count_loss(correct_rate=0.8, incorrect_rate=0.2)

num_epochs = 1  # run for 1 epoch - each data sample is seen only once
num_steps = 25  # run for 25 time steps

# record loss and accuracy over iterations
loss_hist = []
acc_hist = []

# limit number of iterations, set to 0 to disable
iteration_limit = 150

# training loop
for epoch in range(num_epochs):
  for i, (data, targets) in enumerate(iter(train_loader)):
    data = data.to(device)
    targets = targets.to(device)

    net.train()
    spk_rec, _ = net(data)                # forward-pass
    loss_val = loss_fn(spk_rec, targets)  # loss calculation
    optimizer.zero_grad()                 # null gradients
    loss_val.backward()                   # calculate gradients
    optimizer.step()                      # update weights
    loss_hist.append(loss_val.item())     # store loss

    # print every 25 iterations
    if i % 25 == 0:
      net.eval()
      print(f"Epoch {epoch}, iteration {i} \ntraining loss: {loss_val.item():.2f}")

      # check accuracy on a single batch
      acc = SF.accuracy_rate(spk_rec, targets)
      acc_hist.append(acc)
      print(f"Accuracy: {acc * 100:.2f}%\n")

    # limit number of iterations to save time
    if iteration_limit > 0 and i == 150:
        break


Define a function to compute the accuracy score.

In [ ]:
# function to measure accuracy on full test set
def test_accuracy(data_loader, net, num_steps):
  with torch.no_grad():
    total = 0
    acc = 0
    net.eval()

    data_loader = iter(data_loader)
    for data, targets in data_loader:
      data = data.to(device)
      targets = targets.to(device)
      spk_rec, _ = net(data)

      acc += SF.accuracy_rate(spk_rec, targets) * spk_rec.size(1)
      total += spk_rec.size(1)

  return acc / total

See how our model did.

In [ ]:
print(f"Test set accuracy: {test_accuracy(test_loader, net, num_steps) * 100:.3f}%")

# Conclusion

Delta neurons have a lot of potential (no pun intended) and are under active research. It could result in more efficient machine learning and AI models that use far less power. The possible applications of delta neurons and SNNs as a whole remain to be seen.

# References

* [1] Daniel Niel et al. "[Delta Networks for Optimized Recurrent Network Computation](https://proceedings.mlr.press/v70/neil17a/neil17a.pdf)" (2016)
* [2] *Neuronal Dynamics*. [Chapter 5: Nonlinear Integrate-and-Fire Models](https://neuronaldynamics.epfl.ch/online/Ch5.html)